# 🎮 Steam Nexus: Data Integration Pipeline (ERD-Based)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Paths to cleaned datasets
PATH_GAMES_CLEAN = "../data/processed/steam_games_cleaned_v1.csv"
PATH_REVIEWS_CLEAN = "../data/processed/steam_reviews_cleaned_v1.csv"

print("✅ Libraries loaded and paths configured.")

✅ Libraries loaded and paths configured.


## 🔍 1. Cargar Datos Limpios de Games y Reviews

In [2]:
# Load games dataset
print("📊 Loading games dataset...")
df_games = pd.read_csv(PATH_GAMES_CLEAN, low_memory=False)
print(f"✅ Games dataset loaded: {df_games.shape[0]} rows × {df_games.shape[1]} columns")
print(f"📋 Games columns: {list(df_games.columns)}")
display(df_games.head(3))

📊 Loading games dataset...
✅ Games dataset loaded: 122610 rows × 10 columns
📋 Games columns: ['AppID', 'Name', 'Price', 'Genres', 'Tags', 'Positive', 'Negative', 'Release date', 'About the game', 'Developers']


,AppID,Name,Price,Genres,Tags,Positive,Negative,Release date,About the game,Developers
0,2539430,Black Dragon Mage Playtest,0.00,other,[],0,0,2023-08-01,0,unknown
1,496350,Supipara - Chapter 1 Spring Has Come!,5.24,adventure,"adventure,visual novel,anime,cute",252,3,2016-07-29,0,minori
2,1034400,Mystery Solitaire The Black Raven,4.99,casual,"casual,card game,solitaire,puzzle,hidden objec...",21,3,2019-05-06,0,somer games


In [3]:
# Load reviews dataset
print("\n📊 Loading reviews dataset...")
df_reviews = pd.read_csv(PATH_REVIEWS_CLEAN, low_memory=False)
print(f"✅ Reviews dataset loaded: {df_reviews.shape[0]} rows × {df_reviews.shape[1]} columns")
print(f"📋 Reviews columns: {list(df_reviews.columns)}")
display(df_reviews.head(3))


📊 Loading reviews dataset...
✅ Reviews dataset loaded: 6409615 rows × 5 columns
📋 Reviews columns: ['app_id', 'app_name', 'review_text', 'review_score', 'review_votes']


,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,This will be more of a ''my experience with th...,1,1
2,10,Counter-Strike,This game saved my virginity.,1,0


## 🎯 2. Revisar Relaciones del ERD

In [4]:
print("📋 Entity-Relationship Diagram (ERD) Analysis:")
print("==========================================")
print("GAMES ||--o{ REVIEWS : 'receives'")
print("")
print("Primary Keys:")
print("- GAMES.AppID (Primary Key)")
print("- REVIEWS.app_id (Foreign Key → GAMES.AppID)")
print("")
print("Key Relationships:")
print("- GAMES.AppID = REVIEWS.app_id")
print("")
print("Available keys in datasets:")
print(f"- Games has 'AppID': {'AppID' in df_games.columns}")
print(f"- Reviews has 'app_id': {'app_id' in df_reviews.columns}")
print("")
print("Data types:")
print(f"- Games.AppID type: {df_games['AppID'].dtype}")
print(f"- Reviews.app_id type: {df_reviews['app_id'].dtype}")

📋 Entity-Relationship Diagram (ERD) Analysis:
GAMES ||--o{ REVIEWS : 'receives'

Primary Keys:
- GAMES.AppID (Primary Key)
- REVIEWS.app_id (Foreign Key → GAMES.AppID)

Key Relationships:
- GAMES.AppID = REVIEWS.app_id

Available keys in datasets:
- Games has 'AppID': True
- Reviews has 'app_id': True

Data types:
- Games.AppID type: int64
- Reviews.app_id type: int64


## 🔧 3. Normalizar Claves y Formatos de Unión

In [5]:
# Ensure key columns are properly typed for joining
print("🔧 Normalizing key columns...")

# Convert AppID and app_id to int64 for consistent joining
df_games['AppID'] = df_games['AppID'].astype('int64')
df_reviews['app_id'] = df_reviews['app_id'].astype('int64')

print("✅ Key columns normalized:")
print(f"- Games.AppID: {df_games['AppID'].dtype}")
print(f"- Reviews.app_id: {df_reviews['app_id'].dtype}")

# Check for any null values in key columns
games_null_keys = df_games['AppID'].isnull().sum()
reviews_null_keys = df_reviews['app_id'].isnull().sum()

print(f"\n🔍 Null values in key columns:")
print(f"- Games.AppID nulls: {games_null_keys}")
print(f"- Reviews.app_id nulls: {reviews_null_keys}")

# Check unique values in keys
games_unique = df_games['AppID'].nunique()
reviews_unique = df_reviews['app_id'].nunique()

print(f"\n📊 Unique values in key columns:")
print(f"- Games.AppID unique: {games_unique}")
print(f"- Reviews.app_id unique: {reviews_unique}")

# Check overlap between the two datasets
games_appids = set(df_games['AppID'].unique())
reviews_appids = set(df_reviews['app_id'].unique())
overlap = games_appids.intersection(reviews_appids)

print(f"- Overlapping AppIDs: {len(overlap)}")
print(f"- Games without reviews: {len(games_appids - overlap)}")
print(f"- Reviews without games: {len(reviews_appids - overlap)}")

🔧 Normalizing key columns...
✅ Key columns normalized:
- Games.AppID: int64
- Reviews.app_id: int64

🔍 Null values in key columns:
- Games.AppID nulls: 0
- Reviews.app_id nulls: 0

📊 Unique values in key columns:
- Games.AppID unique: 122610
- Reviews.app_id unique: 9972
- Overlapping AppIDs: 8755
- Games without reviews: 113855
- Reviews without games: 1217


## 🔗 4. Unir Datasets Según Modelo Relacional

In [8]:
print("🔗 Performing ERD-based join...")
print("Relationship: GAMES ||--o{ REVIEWS")
print("Join type: LEFT JOIN (keep all games, even those without reviews)")
print("Join keys: GAMES.AppID = REVIEWS.app_id")
print("")

# Perform the join according to ERD
df_integrated = pd.merge(
    df_games,
    df_reviews,
    left_on='AppID',
    right_on='app_id',
    how='left'  # Left join to keep all games
)

print("✅ Join completed!")
print(f"📊 Integrated dataset shape: {df_integrated.shape[0]} rows × {df_integrated.shape[1]} columns")
print(f"📋 Integrated columns: {list(df_integrated.columns)}")

# Check the join results
games_with_reviews = df_integrated['app_id'].notna().sum()
games_without_reviews = df_integrated['app_id'].isna().sum()
total_games = len(df_integrated)

print(f"\n📈 Join Statistics:")
print(f"- Total games: {total_games}")
print(f"- Games with reviews: {games_with_reviews} ({games_with_reviews/total_games*100:.1f}%)")
print(f"- Games without reviews: {games_without_reviews} ({games_without_reviews/total_games*100:.1f}%)")
print(f"- Total reviews: {len(df_reviews)}")

# Show sample of integrated data
print("📋 Sample of integrated data:")
display(df_integrated.head(3))

🔗 Performing ERD-based join...
Relationship: GAMES ||--o{ REVIEWS
Join type: LEFT JOIN (keep all games, even those without reviews)
Join keys: GAMES.AppID = REVIEWS.app_id

✅ Join completed!
📊 Integrated dataset shape: 5935719 rows × 15 columns
📋 Integrated columns: ['AppID', 'Name', 'Price', 'Genres', 'Tags', 'Positive', 'Negative', 'Release date', 'About the game', 'Developers', 'app_id', 'app_name', 'review_text', 'review_score', 'review_votes']

📈 Join Statistics:
- Total games: 5935719
- Games with reviews: 5821864 (98.1%)
- Games without reviews: 113855 (1.9%)
- Total reviews: 6409615
📋 Sample of integrated data:


,AppID,Name,Price,Genres,Tags,Positive,Negative,Release date,About the game,Developers,app_id,app_name,review_text,review_score,review_votes
0,2539430,Black Dragon Mage Playtest,0.00,other,[],0,0,2023-08-01,0,unknown,NaN,NaN,NaN,NaN,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,5.24,adventure,"adventure,visual novel,anime,cute",252,3,2016-07-29,0,minori,NaN,NaN,NaN,NaN,NaN
2,1034400,Mystery Solitaire The Black Raven,4.99,casual,"casual,card game,solitaire,puzzle,hidden objec...",21,3,2019-05-06,0,somer games,NaN,NaN,NaN,NaN,NaN


## ✅ 5. Validar la Unión y Revisar Registros

In [10]:
# Validate the integrated dataset
print("✅ Validating integrated dataset...")

# Check for data integrity
null_summary = df_integrated.isnull().sum()
null_summary = null_summary[null_summary > 0]

print("📊 Null values in integrated dataset:")
if len(null_summary) > 0:
    for col, count in null_summary.items():
        percentage = (count / len(df_integrated)) * 100
        print(f"- {col}: {count} ({percentage:.1f}%)")
else:
    print("No null values found!")

# Check that key relationships are maintained
games_count = df_integrated['AppID'].nunique()
reviews_count = df_integrated['app_id'].notna().sum()

print(f"\n🔗 Relationship validation:")
print(f"- Unique games: {games_count}")
print(f"- Total reviews: {reviews_count}")
print(f"- Average reviews per game: {reviews_count/games_count:.2f}")

# Check data types
print(f"\n📋 Data types in integrated dataset:")
for col in df_integrated.columns:
    print(f"- {col}: {df_integrated[col].dtype}")

# Sample validation - check that game names match between datasets
sample_with_reviews = df_integrated[df_integrated['app_id'].notna()].head(5)
print("🔍 Sample validation (games with reviews):")
for idx, row in sample_with_reviews.iterrows():
    game_name_games = row['Name']
    game_name_reviews = row['app_name']
    match = "✓" if game_name_games == game_name_reviews else "✗"
    print(f"{match} AppID {row['AppID']}: '{game_name_games}' vs '{game_name_reviews}'")

print("\n✅ Validation completed!")

✅ Validating integrated dataset...
📊 Null values in integrated dataset:
- Name: 61 (0.0%)
- app_id: 113855 (1.9%)
- app_name: 113855 (1.9%)
- review_text: 113855 (1.9%)
- review_score: 113855 (1.9%)
- review_votes: 113855 (1.9%)

🔗 Relationship validation:
- Unique games: 122610
- Total reviews: 5821864
- Average reviews per game: 47.48

📋 Data types in integrated dataset:
- AppID: int64
- Name: str
- Price: float64
- Genres: str
- Tags: str
- Positive: int64
- Negative: int64
- Release date: str
- About the game: int64
- Developers: str
- app_id: float64
- app_name: str
- review_text: str
- review_score: float64
- review_votes: float64
🔍 Sample validation (games with reviews):
✓ AppID 328460: 'Redline' vs 'Redline'
✓ AppID 328460: 'Redline' vs 'Redline'
✓ AppID 328460: 'Redline' vs 'Redline'
✓ AppID 328460: 'Redline' vs 'Redline'
✓ AppID 328460: 'Redline' vs 'Redline'

✅ Validation completed!


## 💾 6. Guardar Dataset Combinado

In [12]:
import os

# Create output directory if it doesn't exist
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

# Save the integrated dataset
output_path = os.path.join(output_dir, "steam_games_reviews_integrated_v1.csv")
df_integrated.to_csv(output_path, index=False)

print("💾 Saving integrated dataset...")
print(f"📍 Location: {output_path}")
print(f"📊 Final records: {len(df_integrated)}")
print(f"📋 Columns: {len(df_integrated.columns)}")
print(f"💾 File size: {os.path.getsize(output_path) / (1024**2):.2f} MB")

# Also save a sample for quick analysis
sample_path = os.path.join(output_dir, "steam_games_reviews_integrated_sample_v1.csv")
df_integrated.head(900000).to_csv(sample_path, index=False)
print(f"📋 Sample saved: {sample_path} ({len(df_integrated.head(900000))} records)")

print("\n🎉 Integration pipeline completed successfully!")
print("📈 Ready for analysis and modeling phases.")

💾 Saving integrated dataset...
📍 Location: ../data/processed\steam_games_reviews_integrated_v1.csv
📊 Final records: 5935719
📋 Columns: 15
💾 File size: 3567.63 MB
📋 Sample saved: ../data/processed\steam_games_reviews_integrated_sample_v1.csv (900000 records)

🎉 Integration pipeline completed successfully!
📈 Ready for analysis and modeling phases.
